# Comprehensive Supervision Demo: Smart Traffic Analytics System

This notebook demonstrates the full capabilities of the Supervision library through a comprehensive traffic analytics system.

## Features Demonstrated

1. **Multi-object Detection & Tracking** - Object detection with ByteTracker
2. **Zone Analytics** - Entry/exit counting and time-in-zone analysis
3. **Speed Estimation** - Using perspective transformation
4. **Movement Heatmaps** - Visualization of object movement patterns
5. **Rich Annotations** - Multiple annotation types (boxes, labels, traces)
6. **Video Processing** - Full video pipeline with VideoSink
7. **Metrics Collection** - Analytics export to JSON/CSV
8. **Real-time Performance** - FPS monitoring and optimization

Let's start by importing the necessary libraries and setting up our demo environment.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import json
from pathlib import Path
from collections import defaultdict, deque
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
import time

import supervision as sv

# Set up matplotlib for inline plots
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

print(f"Supervision version: {sv.__version__}")
print("Demo setup complete!")

## 1. Core Components Setup

Let's set up the core components of our analytics system:

In [ ]:
# Import our comprehensive demo classes
import sys
sys.path.append('.')

from demo_comprehensive import (
    AnalyticsConfig, 
    SmartTrafficAnalytics, 
    create_demo_video,
    create_demo_model
)

# Create configuration
config = AnalyticsConfig(
    confidence_threshold=0.3,
    real_world_width=50.0,  # 50 meter road
    real_world_height=25.0,  # 25 meter section
)

print("Analytics configuration:")
print(f"- Confidence threshold: {config.confidence_threshold}")
print(f"- Real world dimensions: {config.real_world_width}m x {config.real_world_height}m")
print(f"- Entry zones: {len(config.entry_zones)}")
print(f"- Exit zones: {len(config.exit_zones)}")

## 2. Detection Models Integration

Supervision supports multiple model backends. Let's demonstrate the flexibility:

In [ ]:
# For the demo, we'll use a mock model
# In real applications, you would use:
# - Ultralytics YOLO: model = YOLO("yolov8n.pt")
# - Roboflow Inference: model = get_model(model_id="...", api_key="...")
# - Transformers: model = pipeline("object-detection", model="...")

model = create_demo_model()

print("Model setup complete!")
print(f"Supported classes: {model.class_names}")

# Test the model on a sample frame
test_frame = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)
test_detections = model(test_frame)

print(f"\nTest detection results:")
print(f"- Number of detections: {len(test_detections)}")
print(f"- Confidence range: {test_detections.confidence.min():.3f} - {test_detections.confidence.max():.3f}")
print(f"- Classes detected: {np.unique(test_detections.class_id)}")

## 3. Video Creation and Processing

Let's create a demo video and process it with our analytics system:

In [ ]:
# Create demo video
demo_video_path = "demo_video.mp4"
create_demo_video(demo_video_path, duration=10, fps=30)

# Get video info
video_info = sv.VideoInfo.from_video_path(demo_video_path)
print(f"Video info:")
print(f"- Resolution: {video_info.width}x{video_info.height}")
print(f"- FPS: {video_info.fps}")
print(f"- Total frames: {video_info.total_frames}")
print(f"- Duration: {video_info.total_frames / video_info.fps:.1f} seconds")

## 4. Analytics System Initialization

Now let's initialize our comprehensive analytics system:

In [ ]:
# Initialize analytics system
analytics = SmartTrafficAnalytics(config)

print("Analytics system initialized with:")
print(f"- Tracker: ByteTrack")
print(f"- Speed estimator: Perspective transformation based")
print(f"- Zone analyzer: {len(analytics.zone_analyzer.entry_zones)} entry zones, {len(analytics.zone_analyzer.exit_zones)} exit zones")
print(f"- Annotators: Box, Label, Trace, Heatmap, Zone")

# Show the zone configuration
print("\nZone configuration:")
for i, zone in enumerate(config.entry_zones):
    print(f"Entry Zone {i}: {zone.tolist()}")
for i, zone in enumerate(config.exit_zones):
    print(f"Exit Zone {i}: {zone.tolist()}")

## 5. Frame-by-Frame Processing Demo

Let's process a few frames to see the system in action:

In [ ]:
# Process first few frames
frame_generator = sv.get_video_frames_generator(demo_video_path)
processed_frames = []
frame_analytics_list = []

for frame_idx, frame in enumerate(frame_generator):
    if frame_idx >= 5:  # Process first 5 frames for demo
        break
        
    timestamp = frame_idx / video_info.fps
    
    # Run detection
    detections = model(frame)
    
    # Process with analytics
    annotated_frame, frame_analytics = analytics.process_frame(
        frame, detections, timestamp
    )
    
    processed_frames.append(annotated_frame)
    frame_analytics_list.append(frame_analytics)
    
    print(f"Frame {frame_idx}: {frame_analytics['detection_count']} detections, {frame_analytics['active_tracks']} active tracks")

print(f"\nProcessed {len(processed_frames)} frames")
print(f"Total unique tracks: {len(analytics.metrics)}")

## 6. Visualization of Results

Let's visualize the processed frames and analytics:

In [ ]:
# Show processed frames
fig, axes = plt.subplots(1, min(3, len(processed_frames)), figsize=(15, 5))
if len(processed_frames) == 1:
    axes = [axes]

for i, (frame, ax) in enumerate(zip(processed_frames[:3], axes)):
    # Convert BGR to RGB for matplotlib
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    ax.imshow(frame_rgb)
    ax.set_title(f"Frame {i}")
    ax.axis('off')

plt.tight_layout()
plt.show()

# Show analytics data
print("\nFrame Analytics:")
for i, analytics_data in enumerate(frame_analytics_list):
    print(f"Frame {i}:")
    print(f"  - Timestamp: {analytics_data['timestamp']:.2f}s")
    print(f"  - Detections: {analytics_data['detection_count']}")
    print(f"  - Active tracks: {analytics_data['active_tracks']}")
    print(f"  - Speeds: {analytics_data['speeds']}")
    print()

## 7. Advanced Analytics Features

Let's explore the advanced analytics capabilities:

In [ ]:
# Track metrics analysis
if analytics.metrics:
    print("Track Metrics Summary:")
    print("=" * 50)
    
    for track_id, metric in analytics.metrics.items():
        print(f"Track ID {track_id}:")
        print(f"  - Class: {metric.class_id}")
        print(f"  - Lifetime: {metric.lifetime:.2f}s")
        print(f"  - Total detections: {metric.total_detections}")
        print(f"  - Average speed: {metric.average_speed:.1f} km/h")
        print(f"  - Zone entries: {metric.zone_entries}")
        print(f"  - Zone exits: {metric.zone_exits}")
        print(f"  - Position history: {len(metric.positions)} points")
        print()
else:
    print("No tracks detected yet")

# Analytics over time
if frame_analytics_list:
    timestamps = [data['timestamp'] for data in frame_analytics_list]
    detection_counts = [data['detection_count'] for data in frame_analytics_list]
    active_tracks = [data['active_tracks'] for data in frame_analytics_list]
    
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(timestamps, detection_counts, 'b-o', label='Detections')
    plt.plot(timestamps, active_tracks, 'r-s', label='Active Tracks')
    plt.xlabel('Time (s)')
    plt.ylabel('Count')
    plt.title('Detections and Tracks Over Time')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    all_speeds = []
    speed_times = []
    for data in frame_analytics_list:
        for track_id, speed in data['speeds'].items():
            if speed > 0:
                all_speeds.append(speed)
                speed_times.append(data['timestamp'])
    
    if all_speeds:
        plt.scatter(speed_times, all_speeds, alpha=0.6)
        plt.xlabel('Time (s)')
        plt.ylabel('Speed (km/h)')
        plt.title('Speed Measurements Over Time')
        plt.grid(True)
    else:
        plt.text(0.5, 0.5, 'No speed data available', transform=plt.gca().transAxes, 
                ha='center', va='center')
        plt.title('Speed Measurements Over Time')
    
    plt.tight_layout()
    plt.show()

## 8. Annotation Showcase

Let's demonstrate different annotation types available in Supervision:

In [ ]:
# Create a test frame with mock detections for annotation showcase
test_frame = np.ones((400, 600, 3), dtype=np.uint8) * 50  # Dark gray background

# Create mock detections
mock_detections = sv.Detections(
    xyxy=np.array([
        [100, 100, 200, 200],
        [300, 150, 400, 250],
        [150, 250, 250, 350]
    ]),
    confidence=np.array([0.9, 0.7, 0.8]),
    class_id=np.array([0, 1, 0]),
    tracker_id=np.array([1, 2, 3])
)

# Different annotation styles
annotators = {
    'Box': sv.BoxAnnotator(color_lookup=sv.ColorLookup.TRACK),
    'Round Box': sv.RoundBoxAnnotator(color_lookup=sv.ColorLookup.TRACK),
    'Corner': sv.BoxCornerAnnotator(color_lookup=sv.ColorLookup.TRACK),
    'Circle': sv.CircleAnnotator(color_lookup=sv.ColorLookup.TRACK),
    'Dot': sv.DotAnnotator(color_lookup=sv.ColorLookup.TRACK),
    'Triangle': sv.TriangleAnnotator(color_lookup=sv.ColorLookup.TRACK)
}

# Create grid of annotations
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, (name, annotator) in enumerate(annotators.items()):
    if i >= len(axes):
        break
        
    annotated = annotator.annotate(test_frame.copy(), mock_detections)
    
    # Add labels
    label_annotator = sv.LabelAnnotator(color_lookup=sv.ColorLookup.TRACK)
    labels = [f"Track {tid}" for tid in mock_detections.tracker_id]
    annotated = label_annotator.annotate(annotated, mock_detections, labels)
    
    # Convert BGR to RGB
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    
    axes[i].imshow(annotated_rgb)
    axes[i].set_title(f'{name} Annotator')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

print("Annotation types demonstrated:")
for name in annotators.keys():
    print(f"- {name}Annotator")

## 9. Zone Analytics Visualization

Let's visualize the zone-based analytics:

In [ ]:
# Create visualization of zones
zone_frame = np.ones((400, 600, 3), dtype=np.uint8) * 30

# Draw entry zones in green
for i, zone_polygon in enumerate(config.entry_zones):
    # Adjust zone coordinates for our visualization frame
    adjusted_polygon = zone_polygon.copy()
    adjusted_polygon[:, 0] = np.clip(adjusted_polygon[:, 0], 0, 600)
    adjusted_polygon[:, 1] = np.clip(adjusted_polygon[:, 1], 0, 400)
    
    cv2.fillPoly(zone_frame, [adjusted_polygon.astype(np.int32)], (0, 255, 0))  # Green
    
    # Add zone label
    center = np.mean(adjusted_polygon, axis=0).astype(int)
    cv2.putText(zone_frame, f'Entry {i}', tuple(center), 
               cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

# Draw exit zones in red
for i, zone_polygon in enumerate(config.exit_zones):
    adjusted_polygon = zone_polygon.copy()
    adjusted_polygon[:, 0] = np.clip(adjusted_polygon[:, 0], 0, 600)
    adjusted_polygon[:, 1] = np.clip(adjusted_polygon[:, 1], 0, 400)
    
    cv2.fillPoly(zone_frame, [adjusted_polygon.astype(np.int32)], (0, 0, 255))  # Red
    
    center = np.mean(adjusted_polygon, axis=0).astype(int)
    cv2.putText(zone_frame, f'Exit {i}', tuple(center), 
               cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

# Draw speed estimation area
speed_polygon = config.speed_source_points.copy()
speed_polygon[:, 0] = np.clip(speed_polygon[:, 0], 0, 600)
speed_polygon[:, 1] = np.clip(speed_polygon[:, 1], 0, 400)
cv2.polylines(zone_frame, [speed_polygon.astype(np.int32)], True, (255, 255, 0), 3)  # Yellow

# Add legend
cv2.putText(zone_frame, 'Green: Entry Zones', (10, 30), 
           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
cv2.putText(zone_frame, 'Red: Exit Zones', (10, 60), 
           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
cv2.putText(zone_frame, 'Yellow: Speed Zone', (10, 90), 
           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

# Display
plt.figure(figsize=(10, 6))
zone_frame_rgb = cv2.cvtColor(zone_frame, cv2.COLOR_BGR2RGB)
plt.imshow(zone_frame_rgb)
plt.title('Zone Configuration for Analytics')
plt.axis('off')
plt.show()

print("Zone Analytics Features:")
print("- Entry/Exit counting")
print("- Time spent in zones")
print("- Zone occupancy monitoring")
print("- Cross-zone traffic flow analysis")

## 10. Dataset Operations Demo

Let's demonstrate Supervision's dataset handling capabilities:

In [ ]:
# Create a mock dataset for demonstration
# In real applications, you would load from actual dataset files

print("Dataset Operations Demo")
print("=" * 30)

# Mock dataset creation
print("\n1. Dataset Loading:")
print("   - sv.DetectionDataset.from_coco()")
print("   - sv.DetectionDataset.from_yolo()")
print("   - sv.DetectionDataset.from_pascal_voc()")

# Mock detection dataset
class MockDetectionDataset:
    def __init__(self, size=100):
        self.size = size
        self.classes = ['vehicle', 'person', 'bicycle']
    
    def __len__(self):
        return self.size
    
    def split(self, split_ratio=0.8):
        train_size = int(self.size * split_ratio)
        test_size = self.size - train_size
        return MockDetectionDataset(train_size), MockDetectionDataset(test_size)

# Demonstrate dataset operations
dataset = MockDetectionDataset(1000)
print(f"\n2. Dataset Info:")
print(f"   - Total samples: {len(dataset)}")
print(f"   - Classes: {dataset.classes}")

print(f"\n3. Dataset Splitting:")
train_dataset, test_dataset = dataset.split(split_ratio=0.8)
print(f"   - Train: {len(train_dataset)} samples")
print(f"   - Test: {len(test_dataset)} samples")

test_dataset, val_dataset = test_dataset.split(split_ratio=0.5)
print(f"   - Test: {len(test_dataset)} samples")
print(f"   - Validation: {len(val_dataset)} samples")

print(f"\n4. Format Conversion:")
print("   - dataset.as_coco() - Export to COCO format")
print("   - dataset.as_yolo() - Export to YOLO format")
print("   - dataset.as_pascal_voc() - Export to Pascal VOC format")

print(f"\n5. Dataset Utilities:")
print("   - sv.mask_to_rle() - Convert masks to RLE")
print("   - sv.rle_to_mask() - Convert RLE to masks")
print("   - sv.get_coco_class_index_mapping() - COCO class mapping")

## 11. Performance Monitoring

Let's demonstrate the performance monitoring capabilities:

In [ ]:
# Performance monitoring demo
fps_monitor = sv.FPSMonitor()
processing_times = []

print("Performance Monitoring Demo")
print("=" * 30)

# Simulate processing frames
for i in range(10):
    start_time = time.time()
    
    # Simulate frame processing
    fps_monitor.tick()
    
    # Mock processing (detection + tracking + analytics)
    time.sleep(0.033)  # Simulate ~30 FPS processing
    
    processing_time = time.time() - start_time
    processing_times.append(processing_time)
    
    current_fps = fps_monitor.fps
    print(f"Frame {i+1}: {current_fps:.1f} FPS, Processing time: {processing_time*1000:.1f}ms")

# Performance statistics
avg_processing_time = np.mean(processing_times)
max_processing_time = np.max(processing_times)
min_processing_time = np.min(processing_times)

print(f"\nPerformance Summary:")
print(f"- Average FPS: {fps_monitor.fps:.1f}")
print(f"- Average processing time: {avg_processing_time*1000:.1f}ms")
print(f"- Min processing time: {min_processing_time*1000:.1f}ms")
print(f"- Max processing time: {max_processing_time*1000:.1f}ms")

# Visualize performance
plt.figure(figsize=(10, 4))
plt.plot(range(1, len(processing_times)+1), np.array(processing_times)*1000, 'b-o')
plt.axhline(y=avg_processing_time*1000, color='r', linestyle='--', label=f'Average ({avg_processing_time*1000:.1f}ms)')
plt.xlabel('Frame Number')
plt.ylabel('Processing Time (ms)')
plt.title('Frame Processing Performance')
plt.legend()
plt.grid(True)
plt.show()

## 12. Data Export and Analytics

Finally, let's demonstrate the data export capabilities:

In [ ]:
# Export analytics data
output_dir = Path("demo_analytics_output")
output_dir.mkdir(exist_ok=True)

# Export analytics (if we have data)
if hasattr(analytics, 'metrics') and analytics.metrics:
    analytics.export_analytics(output_dir)
    
    # Check if files were created
    json_file = output_dir / "analytics_summary.json"
    csv_file = output_dir / "frame_analytics.csv"
    
    if json_file.exists():
        with open(json_file, 'r') as f:
            summary_data = json.load(f)
        print("Analytics Summary:")
        print(f"- Total tracks: {summary_data['total_tracks']}")
        print(f"- Total frames: {summary_data['total_frames']}")
        print(f"- Export files created:")
        print(f"  - {json_file}")
        print(f"  - {csv_file}")
    else:
        print("No analytics data available for export")
else:
    print("No tracking data available for export")

# Demonstrate other export formats
print("\nSupported Export Formats:")
print("- JSON: Complete analytics summary")
print("- CSV: Frame-by-frame data")
print("- sv.CSVSink: Real-time CSV logging")
print("- sv.JSONSink: Real-time JSON logging")

# Example of using data sinks
print("\nData Sink Examples:")
csv_sink_example = "sv.CSVSink('detections.csv')"
json_sink_example = "sv.JSONSink('detections.json')"

print(f"CSV Sink: {csv_sink_example}")
print(f"JSON Sink: {json_sink_example}")
print("These can be used to log detection data in real-time during video processing.")

## 13. Summary and Key Features

This comprehensive demo has showcased the full capabilities of the Supervision library:

In [ ]:
print("🎯 SUPERVISION COMPREHENSIVE DEMO SUMMARY")
print("=" * 50)

features_demonstrated = {
    "🔍 Object Detection": [
        "Multiple model backend support (YOLO, Transformers, Inference)",
        "Detection filtering by confidence and class",
        "Non-maximum suppression utilities"
    ],
    "📍 Object Tracking": [
        "ByteTracker integration",
        "Consistent ID assignment",
        "Track lifecycle management"
    ],
    "🎨 Rich Annotations": [
        "Box, RoundBox, Corner, Circle, Dot, Triangle annotators",
        "Label and trace annotations",
        "Color lookup strategies",
        "Heatmap visualizations"
    ],
    "📊 Zone Analytics": [
        "PolygonZone and LineZone monitoring",
        "Entry/exit counting",
        "Time-in-zone analysis",
        "Zone occupancy tracking"
    ],
    "⚡ Speed Estimation": [
        "Perspective transformation",
        "Real-world coordinate mapping",
        "Movement speed calculation"
    ],
    "🎬 Video Processing": [
        "VideoInfo for metadata extraction",
        "VideoSink for output generation",
        "Frame-by-frame processing",
        "FPS monitoring"
    ],
    "📁 Dataset Operations": [
        "COCO, YOLO, Pascal VOC format support",
        "Dataset splitting and merging",
        "Format conversion utilities",
        "Mask and RLE conversions"
    ],
    "📈 Metrics & Export": [
        "Confusion matrix calculation",
        "Mean Average Precision (mAP)",
        "CSV and JSON data export",
        "Real-time data sinks"
    ],
    "🔧 Utilities": [
        "Image processing utilities",
        "Geometry operations",
        "Color palette management",
        "File handling utilities"
    ]
}

for category, features in features_demonstrated.items():
    print(f"\n{category}:")
    for feature in features:
        print(f"  ✓ {feature}")

print("\n" + "=" * 50)
print("🚀 Next Steps:")
print("  1. Replace mock model with real YOLO/Inference model")
print("  2. Configure zones for your specific use case")
print("  3. Adjust parameters for optimal performance")
print("  4. Integrate with your video processing pipeline")
print("  5. Set up real-time analytics dashboard")

print("\n📚 Documentation: https://supervision.roboflow.com/")
print("🐛 Issues: https://github.com/roboflow/supervision/issues")
print("💬 Discussions: https://github.com/roboflow/supervision/discussions")